┌─────────────────────────────────────────────────────────┐
│                    DATA LAYER                           │
│  (Your existing pipeline — historical + live streaming) │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  FEATURE LAYER                          │
│  (Alignment, resampling, all engineered features)       │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  SIGNAL LAYER                           │
│  (Individual setups A/B/C/D — each independently)      │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  RISK LAYER                             │
│  (Position sizing, max drawdown, correlation limits)    │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│               EXECUTION LAYER                           │
│  (Order routing, slippage, Binance API)                 │
└─────────────────────────────────────────────────────────┘

In [ ]:
from orderflow import futures_agg_trades, get_oi, get_mark_price_klines, spot_agg_trades, get_premium_index_klines
import pandas as pd
import numpy as np

symbol = "BTCUSDT"
start = pd.Timestamp("2026-06-07 12:00")
end   = pd.Timestamp("2026-06-10 18:00")
timeframe = "5min" 

# Funding rate requires pre-warming hence the start - pd.Timedelta(days=1))
dates = [d.strftime("%Y-%m-%d")
         for d in pd.date_range((start - pd.Timedelta(days=1)).normalize(), end.normalize(), freq="D")]
all = lambda fn: pd.concat([fn(symbol, d) for d in dates]).sort_index()

fut_trades = all(futures_agg_trades)
oi         = all(get_oi)
context    = all(get_mark_price_klines)
spot       = all(spot_agg_trades)
premium    = all(get_premium_index_klines)

ohlc = context.resample(timeframe).agg({"open": "first", "high": "max", "low": "min", "close": "last"})

fut_cvd  = fut_trades["volume_delta"].resample(timeframe).sum().cumsum()
spot_cvd = spot["volume_delta"].resample(timeframe).sum().cumsum()

I = 0.0001
step = premium.index.to_series().diff().median()
n = int(pd.Timedelta("8h") / step)
w = np.arange(1, n + 1)
p_avg = premium["close"].rolling(n).apply(lambda x: x @ w / w.sum(), raw=True)
funding_rate = p_avg + (I - p_avg).clip(-0.0005, 0.0005)
funding_annualized = (funding_rate * 3 * 365).resample(timeframe).last()

oi_tf = oi["sum_open_interest"].resample(timeframe).last()

In [8]:
combined = pd.concat([
    fut_cvd.rename("fut_cumulative_volume_delta"),
    spot_cvd.rename("spot_cumulative_volume_delta"),
    ohlc,
    oi_tf.rename("sum_open_interest"),
    funding_annualized.rename("funding_rate_annualized"),
], axis=1)

combined = combined.loc[start.floor(timeframe):end]

for col in ["fut_cumulative_volume_delta", "spot_cumulative_volume_delta"]:
    combined[col] = combined[col] - combined[col].dropna().iloc[0]

combined

,fut_cumulative_volume_delta,spot_cumulative_volume_delta,open,high,low,close,sum_open_interest,funding_rate_annualized
2026-06-07 12:00:00,0.000,0.00000,62598.900000,62617.700000,62540.000000,62603.364572,102029.354,0.023831
2026-06-07 12:05:00,-396.644,-28.58175,62603.350877,62616.200000,62344.091551,62393.000000,102038.958,0.020968
2026-06-07 12:10:00,-1656.429,-127.03931,62396.200000,62417.800000,62042.474601,62055.139595,101849.952,0.020433
2026-06-07 12:15:00,-2500.790,-211.03500,62055.000000,62203.026184,61906.207036,61981.600000,101361.319,0.017237
2026-06-07 12:20:00,-2121.748,-272.85834,61971.687449,62134.100000,61909.100000,61946.500000,100886.177,0.017825
...,...,...,...,...,...,...,...,...
2026-06-10 17:40:00,-3898.978,-1295.30726,61740.383232,61818.700000,61623.328630,61720.255427,99866.481,-0.003091
2026-06-10 17:45:00,-3773.152,-1270.73115,61714.145486,62015.700000,61693.889580,62015.700000,100495.303,0.001481
2026-06-10 17:50:00,-3935.368,-1297.97916,62015.800000,62015.800000,61810.354486,61863.800000,100389.405,0.000225
2026-06-10 17:55:00,-3859.963,-1296.63551,61863.700000,61977.200000,61863.700000,61944.400000,100363.108,0.000071


In [9]:
from orderflow import build_order_flow_chart

my_layout = [
    {"type": "candlestick", "title": "Price Action", "name": "Candlestick", "y_title": "Price (USDT)"},
    {"type": "line", "title": "Open Interest", "column": "sum_open_interest", "name": "OI Coins", "color": "purple", "y_title": "OI"},
    {"type": "delta_bars", "title": "Funding Rate", "column": "funding_rate_annualized", "name": "Funding", "color": "orange", "y_title": "Rate"},
    {"type": "delta_bars", "title": "Spot CVD", "column": "spot_cumulative_volume_delta", "name": "Spot Delta", "y_title": "Delta"},
    {"type": "delta_bars", "title": "Futures CVD", "column": "fut_cumulative_volume_delta", "name": "Futures Delta", "y_title": "Delta"}
]

fig = build_order_flow_chart(combined, f"{start:%Y-%m-%d %H:%M} → {end:%Y-%m-%d %H:%M}", config=my_layout, timeframe=timeframe)
fig.show()